# Association Rule Mining
### Online Retail Transaction Dataset

**Objective:** Apply rule-mining techniques — specifically market basket analysis — to the online retail
transactions dataset in order to uncover products that customers frequently purchase together. I use the
**Apriori algorithm** to mine frequent itemsets and derive association rules (support, confidence, lift,
leverage, conviction), then interpret the results to generate insight into customer purchasing behaviour.


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from collections import Counter
import json

import matplotlib
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
%matplotlib inline

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

In [ ]:
DATA_PATH = 'Online_retail.xlsx' 

raw = pd.read_excel(DATA_PATH, header=None)
print('Raw shape:', raw.shape)
raw.head(10)

## 2. Data Quality Checks

Before mining, we check for missing values and duplicate rows so we can decide how to handle them.

In [ ]:
n_rows_raw = raw.shape[0]
n_fully_null = raw[0].isnull().sum()
dup_rows = raw.duplicated().sum()

print(f"Total rows (transactions): {n_rows_raw}")
print(f"Fully blank rows: {n_fully_null}")
print(f"Exact duplicate rows: {dup_rows}")

**Note on duplicate rows:** many rows are identical to another row (e.g. several different customers each
buying only "mineral water"). These are **not** data-entry duplicates to be dropped — each row is an
independent transaction/customer visit. Removing them would understate the true support of common items and
distort every support/confidence/lift calculation downstream, so we deliberately keep them.

## 3. Data Preprocessing

Association rule mining needs the data in a specific structural form (one row per transaction, item
membership only — not quantity). Steps applied:

1. **Text normalisation** — trim whitespace and lower-case every item name (e.g. `" Mineral Water"` and
   `"mineral water"` must be treated as the same product).
2. **Drop blank tokens** — trailing empty cells from shorter baskets are removed.
3. **Within-basket de-duplication** — if an item appears twice in one row, keep a single occurrence; for
   association rules we only care whether an item is *present* in a basket.
4. **Drop empty baskets** — any transaction left with zero items after cleaning is discarded.


In [ ]:
def clean_transaction(row):
    items = [i.strip().lower() for i in row.split(',')]
    items = [i for i in items if i not in ('', 'nan')]
    return sorted(set(items))   # de-duplicate items within the same basket

transactions_raw = raw[0].dropna().astype(str).tolist()
transactions = [clean_transaction(r) for r in transactions_raw]
transactions = [t for t in transactions if len(t) > 0]

n_transactions = len(transactions)
basket_sizes = [len(t) for t in transactions]
all_items = sorted(set(i for t in transactions for i in t))
n_items = len(all_items)

print(f"Clean transactions: {n_transactions}")
print(f"Unique items: {n_items}")
print(f"Basket size — min: {min(basket_sizes)}, max: {max(basket_sizes)}, "
      f"mean: {np.mean(basket_sizes):.2f}, median: {np.median(basket_sizes):.0f}")

## 4. Exploratory Data Analysis (EDA)

### 4.1 Item Frequency

In [ ]:
item_counts = Counter(i for t in transactions for i in t)
item_freq_df = (pd.DataFrame(item_counts.items(), columns=['item', 'count'])
                 .sort_values('count', ascending=False)
                 .reset_index(drop=True))
item_freq_df['support_pct'] = 100 * item_freq_df['count'] / n_transactions

item_freq_df.head(10)

In [ ]:
top20 = item_freq_df.head(20).iloc[::-1]

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.barh(top20['item'], top20['count'], color='#4C72B0')
ax.set_xlabel('Number of Transactions')
ax.set_title('Top 20 Most Frequently Purchased Items', fontsize=13, fontweight='bold')
for b in bars:
    ax.text(b.get_width() + 10, b.get_y() + b.get_height()/2, f'{int(b.get_width())}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

### 4.2 Basket Size Distribution

In [ ]:
basket_size_counts = Counter(basket_sizes)
sizes = sorted(basket_size_counts.keys())
counts = [basket_size_counts[s] for s in sizes]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(sizes, counts, color='#55A868')
ax.set_xlabel('Basket Size (number of items)')
ax.set_ylabel('Number of Transactions')
ax.set_title('Distribution of Transaction (Basket) Sizes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Basket sizes are right-skewed: most customers buy only a handful of items, with a long tail of larger,
less frequent "big-basket" shops.

### 4.3 Long-Tail Item Popularity

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(item_freq_df)+1), item_freq_df['count'].values, marker='o', markersize=3, color='#C44E52')
ax.set_xlabel('Item Rank')
ax.set_ylabel('Frequency (# transactions)')
ax.set_title('Item Purchase Frequency — Long Tail Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

A small number of items (mineral water, eggs, spaghetti, staples) account for a disproportionate share of
purchases, while most of the 119 items are bought relatively rarely (classic Pareto / long-tail pattern). This
means a fairly **low minimum-support threshold** is needed to surface any multi-item rules at all.

## 5. One-Hot Encoding

Convert the cleaned list-of-lists into a boolean transaction × item matrix — the standard input format
required by the Apriori algorithm.

In [ ]:
item_index = {item: idx for idx, item in enumerate(all_items)}
onehot = np.zeros((n_transactions, n_items), dtype=bool)
for row_idx, t in enumerate(transactions):
    for it in t:
        onehot[row_idx, item_index[it]] = True

onehot_df = pd.DataFrame(onehot, columns=all_items)
print('One-hot matrix shape:', onehot_df.shape)
onehot_df.head()

## 6. Apriori Algorithm — Frequent Itemset Mining

`mlxtend` is the usual off-the-shelf library for this (`mlxtend.frequent_patterns.apriori`), but to keep this
notebook fully self-contained and dependency-free we implement the same **level-wise Apriori** procedure
directly with Pandas/NumPy:

1. Generate candidate 1-itemsets and keep those meeting minimum support.
2. Iteratively **join** frequent k-itemsets into (k+1)-item candidates, **pruning** any candidate that
   contains an infrequent k-subset (the Apriori "anti-monotonicity" property — if any subset of an itemset is
   infrequent, the itemset itself cannot be frequent).
3. Count support for surviving candidates directly against the one-hot matrix and keep those at or above the
   support threshold.
4. Repeat until no new frequent itemsets are found.

**Thresholds** (chosen empirically for this dataset's sparsity — 119 items, ~4 items/basket on average):

In [ ]:
MIN_SUPPORT = 0.02     # ~150+ of 7,501 transactions
MIN_CONFIDENCE = 0.20
MIN_LIFT = 1.0

print(f"min_support={MIN_SUPPORT}, min_confidence={MIN_CONFIDENCE}, min_lift={MIN_LIFT}")

In [ ]:
item_arrays = {item: onehot_df[item].values for item in all_items}

def support_of(itemset):
    """Boolean-AND across item columns; returns (support, boolean mask)."""
    mask = None
    for it in itemset:
        arr = item_arrays[it]
        mask = arr if mask is None else (mask & arr)
    return mask.sum() / n_transactions, mask

def apriori_gen(Lk, k):
    """Generate candidate (k+1)-itemsets from frequent k-itemsets Lk, with subset pruning."""
    candidates = set()
    Lk_list = sorted([tuple(sorted(x)) for x in Lk])
    n = len(Lk_list)
    for i in range(n):
        for j in range(i + 1, n):
            a, b = Lk_list[i], Lk_list[j]
            if a[:k-1] == b[:k-1]:                      # same (k-1)-prefix -> can join
                candidate = tuple(sorted(set(a) | set(b)))
                if len(candidate) == k + 1:
                    subsets_ok = all(frozenset(c) in freq_itemsets
                                      for c in combinations(candidate, k))
                    if subsets_ok:
                        candidates.add(frozenset(candidate))
    return candidates

In [ ]:
freq_itemsets = {}   # frozenset(items) -> support

# ---- Level 1: frequent single items ----
L1 = []
for item in all_items:
    supp, _ = support_of((item,))
    if supp >= MIN_SUPPORT:
        fs = frozenset([item])
        freq_itemsets[fs] = supp
        L1.append(fs)

print('Frequent 1-itemsets:', len(L1))

# ---- Levels 2, 3, ... : iterative join + prune + count ----
current_L = L1
k = 1
while current_L:
    candidates = apriori_gen(current_L, k)
    next_L = []
    for cand in candidates:
        supp, _ = support_of(tuple(cand))
        if supp >= MIN_SUPPORT:
            freq_itemsets[cand] = supp
            next_L.append(cand)
    print(f'Frequent {k+1}-itemsets:', len(next_L))
    if not next_L:
        break
    current_L = next_L
    k += 1

print('\nTotal frequent itemsets found:', len(freq_itemsets))

In [ ]:
freq_itemsets_df = pd.DataFrame([
    {'itemset': ', '.join(sorted(k)), 'size': len(k), 'support': round(v, 4)}
    for k, v in freq_itemsets.items()
]).sort_values(['size', 'support'], ascending=[True, False]).reset_index(drop=True)

freq_itemsets_df.to_csv('frequent_itemsets.csv', index=False)
freq_itemsets_df.head(15)

## 7. Association Rule Generation

From every frequent itemset of size ≥ 2, generate all possible antecedent → consequent splits and compute:

- **Support(A→B)** = P(A ∩ B) — how often the whole rule occurs
- **Confidence(A→B)** = P(B | A) = Support(A∩B) / Support(A) — reliability of the rule
- **Lift(A→B)** = Confidence(A→B) / Support(B) — how much more likely B is, given A, vs. chance
- **Leverage(A→B)** = Support(A∩B) − Support(A)·Support(B) — absolute difference from independence
- **Conviction(A→B)** = (1 − Support(B)) / (1 − Confidence(A→B)) — how much A "implies" B

In [ ]:
rules = []
for itemset, supp in freq_itemsets.items():
    if len(itemset) < 2:
        continue
    items = list(itemset)
    for i in range(1, len(items)):
        for antecedent in combinations(items, i):
            antecedent = frozenset(antecedent)
            consequent = itemset - antecedent

            supp_ante = freq_itemsets.get(antecedent) or support_of(tuple(antecedent))[0]
            supp_cons = freq_itemsets.get(consequent) or support_of(tuple(consequent))[0]

            confidence = supp / supp_ante
            lift = confidence / supp_cons
            leverage = supp - (supp_ante * supp_cons)
            conviction = (1 - supp_cons) / (1 - confidence) if confidence < 1 else np.inf

            if confidence >= MIN_CONFIDENCE and lift >= MIN_LIFT:
                rules.append({
                    'antecedent': ', '.join(sorted(antecedent)),
                    'consequent': ', '.join(sorted(consequent)),
                    'antecedent_support': round(supp_ante, 4),
                    'consequent_support': round(supp_cons, 4),
                    'support': round(supp, 4),
                    'confidence': round(confidence, 4),
                    'lift': round(lift, 4),
                    'leverage': round(leverage, 4),
                    'conviction': round(conviction, 4) if conviction != np.inf else np.inf,
                })

rules_df = pd.DataFrame(rules).sort_values(['lift', 'confidence'], ascending=False).reset_index(drop=True)
rules_df.to_csv('association_rules.csv', index=False)

print(f"Total rules generated (support>={MIN_SUPPORT}, confidence>={MIN_CONFIDENCE}, lift>={MIN_LIFT}): {len(rules_df)}")
rules_df.head(15)

## 8. Results Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(rules_df['support'], rules_df['confidence'], c=rules_df['lift'],
                 cmap='viridis', s=80, edgecolor='k', linewidth=0.4, alpha=0.85)
cbar = plt.colorbar(sc)
cbar.set_label('Lift')
ax.set_xlabel('Support')
ax.set_ylabel('Confidence')
ax.set_title('Association Rules: Support vs. Confidence (colored by Lift)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
top15 = rules_df.sort_values('lift', ascending=False).head(15).iloc[::-1].copy()
top15['label'] = top15['antecedent'] + '  ->  ' + top15['consequent']

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(top15['label'], top15['lift'], color='#8172B2')
ax.set_xlabel('Lift')
ax.set_title('Top 15 Association Rules by Lift', fontsize=13, fontweight='bold')
for b in bars:
    ax.text(b.get_width() + 0.02, b.get_y() + b.get_height()/2, f'{b.get_width():.2f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
## 9. Interview Questions.

** What is Lift and why is it important in Association rules?
** Answer: Lift measures how strongly two items are associated. A lift greater than 1 indicates a positive relation between the items.

** What is Support and Confidence?
** Answer: Support shows how frequently an item appears in the dataset.
** Confidence shows how often a rule is true.

** Limitations of Association Rule Mining?
** Answer: It can generate too many rules.
** It may find irrelevant or misleading relationships.
** It can be computationally expensive for very large datasets.
**
                                      
** 